# R3: CodeBERT로 diff 읽기

**목표:** PR의 diff(바뀐 코드)를 CodeBERT에 넣어 숫자 목록(임베딩)으로 바꾸고, R1 피처에 붙여서(R3) 머지 예측이 좋아지는지, 시간은 얼마나 드는지 잰다.

- 데이터: `r3_sample.parquet` (R2 실험과 **같은 표본**, 학습 3만 개 + 평가 3만 개)
- 분류기: R0~R2와 **같은 XGBoost 설정**, 난수 시드 5개 반복
- 비교 기준: 같은 노트북에서 R1도 다시 돌려서 R3와 나란히 비교한다

**실행 방법:** 위 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 → `런타임 → 모두 실행`.
중간에 끊겨도 다시 `모두 실행`하면 계산해 둔 부분부터 이어서 한다.

In [ ]:
# ==== 설정 (보통 바꿀 필요 없음) ====
import os
DATA_DIR = os.environ.get("R3_DATA_DIR", "/content/drive/MyDrive/aix_r3")  # r3_sample.parquet을 올린 구글 드라이브 폴더
OUT_DIR = os.environ.get("R3_OUT_DIR", DATA_DIR)    # 결과와 중간 저장 파일이 생기는 곳
SEEDS = [0, 1, 2, 3, 4]                             # 같은 실험을 5번 (R0~R2와 같다)
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512      # CodeBERT가 한 번에 읽을 수 있는 최대 토큰 수 (diff 앞부분만 읽는다)
BATCH = 64            # GPU 메모리가 부족하다는 오류가 나면 32나 16으로 줄인다
CHUNK = 2000          # 이만큼 계산할 때마다 드라이브에 저장 (끊겨도 이어서 계산)
PCA_DIM = 128         # R2와 같이 128개 숫자로 줄인다
MOCK = os.environ.get("R3_MOCK") == "1"  # 코드 점검용. True면 CodeBERT 대신 무작위 숫자를 쓴다 (결과 의미 없음)

In [ ]:
# ==== 준비: 구글 드라이브 연결, 라이브러리 설치 ====
import sys, json, time, glob, platform, subprocess
if os.path.exists("/content") and not MOCK:          # Colab일 때만
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost==3.2.0"], check=True)
os.makedirs(OUT_DIR, exist_ok=True)
import numpy as np, pandas as pd, xgboost as xgb, sklearn
from sklearn.metrics import roc_auc_score, average_precision_score
print("xgboost", xgb.__version__, "| scikit-learn", sklearn.__version__, "| CPU 코어", os.cpu_count())
assert xgb.__version__.startswith("3.2"), "xgboost 3.2가 아니면 런타임 → 세션 다시 시작 후 다시 실행"

In [ ]:
# ==== 데이터 불러오기 ====
df = pd.read_parquet(f"{DATA_DIR}/r3_sample.parquet").sort_values("pr_id").reset_index(drop=True)
tr_mask = (df.split == "train").to_numpy()
print("PR 수:", len(df), "| 학습:", tr_mask.sum(), "| 평가:", (~tr_mask).sum())
print("평가 세트 거절 비율:", round(1 - df.y[~tr_mask].mean(), 4), "(0.242여야 한다)")
assert len(df) == 60000 and tr_mask.sum() == 30000

## 1단계: 기준 확인 (R0, R1 다시 돌리기)

블로그에 적힌 값과 같게 나오는지 먼저 확인한다. 같게 나오면 이 노트북 환경이 우리 실험과 같다는 뜻이다.

In [ ]:
PARAMS = dict(n_estimators=400, max_depth=6, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
              tree_method="hist", enable_categorical=True, n_jobs=4, random_state=42, eval_metric="auc")
R0 = ["agent", "lang", "task", "stars", "forks", "repo_prior_prs", "title_len", "body_len"]
R1 = R0 + ["top_ext", "n_files", "additions", "deletions", "log_churn", "n_ext", "has_test", "test_line_share", "doc_file_share"]
tr, te = df[tr_mask].copy(), df[~tr_mask].copy()
for c in ["agent", "lang", "task", "top_ext"]:                # 학습 세트에서 많이 나온 30개만 쓰고 나머지는 other_
    top = tr[c].value_counts().index[:30]
    for d in (tr, te):
        d[c] = pd.Categorical(np.where(d[c].isin(top), d[c], "other_"), categories=list(top) + ["other_"])
y_te = te.y.to_numpy()

def run(X, seed):
    m = xgb.XGBClassifier(**{**PARAMS, "random_state": seed})
    t = time.time(); m.fit(tr[X], tr.y); ts = time.time() - t
    t = time.time(); p = m.predict_proba(te[X])[:, 1]; tp = time.time() - t
    return p, {"auc": roc_auc_score(y_te, p), "prauc_reject": average_precision_score(1 - y_te, 1 - p), "train_sec": ts, "pred_sec": tp}

res, preds = {}, pd.DataFrame({"pr_id": te.pr_id.to_numpy(), "y": y_te})
for name, X in [("R0", R0), ("R1", R1)]:
    res[name] = {}
    for s in SEEDS:
        p, r = run(X, s); res[name][str(s)] = r; preds[f"{name}_p{s}"] = p
    a = np.mean([r["auc"] for r in res[name].values()])
    print(name, "AUC 5번 평균:", round(a, 4), [round(r["auc"], 4) for r in res[name].values()])
REF = {"R0": 0.6798, "R1": 0.7188}   # 블로그 값 (같은 표본, 5번 평균)
for k, v in REF.items():
    a = np.mean([r["auc"] for r in res[k].values()])
    print(("OK  " if abs(a - v) < 0.005 else "확인 필요  ") + f"{k}: 이번 {a:.4f} / 블로그 {v:.4f}")

## 2단계: CodeBERT 임베딩 만들기

CodeBERT는 많은 코드를 미리 읽고 공부한 AI 모델이다. diff를 넣으면 그 뜻을 담은 숫자 768개(임베딩)를 돌려준다.
여기서는 모든 단어 자리의 숫자를 평균 내서(mean pooling) PR 하나당 768개를 만든다.
6만 개를 2,000개씩 나눠 계산하고, 나눈 조각마다 드라이브에 저장한다.

In [ ]:
texts = df["diff"].fillna("").tolist()
if MOCK:
    gpu_name = "MOCK"
    def embed(batch_texts):
        rng = np.random.default_rng(len(batch_texts))
        return rng.normal(size=(len(batch_texts), 768)).astype(np.float32)
else:
    import torch
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    gpu_name = torch.cuda.get_device_name(0) if dev == "cuda" else "CPU"
    print("계산 장치:", gpu_name, "(GPU가 아니면 런타임 유형을 T4 GPU로 바꾼다)")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    enc = AutoModel.from_pretrained(MODEL_NAME).to(dev).eval()
    if dev == "cuda":
        enc = enc.half()   # 16비트로 계산해서 빠르게

    def embed(batch_texts, model=None, device=None, batch=BATCH):
        model = model or enc; device = device or dev
        out = np.zeros((len(batch_texts), model.config.hidden_size), dtype=np.float32)
        order = np.argsort([len(t) for t in batch_texts])          # 길이가 비슷한 것끼리 묶으면 빠르다
        for i in range(0, len(batch_texts), batch):
            idx = order[i:i + batch]
            b = tok([batch_texts[j] for j in idx], truncation=True, max_length=MAX_LENGTH,
                    padding=True, return_tensors="pt").to(device)
            with torch.no_grad():
                h = model(**b).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)                 # 빈칸(패딩)을 뺀 평균
            out[idx] = v.float().cpu().numpy()
        return out

log_path = f"{OUT_DIR}/r3_embed_log.json"
log = json.load(open(log_path)) if os.path.exists(log_path) else {}
for k in range(0, len(texts), CHUNK):
    path = f"{OUT_DIR}/r3_emb_{k:06d}.npy"
    if os.path.exists(path) and str(k) in log:
        continue                                                     # 이미 계산한 조각은 건너뛴다
    t = time.time(); e = embed(texts[k:k + CHUNK])
    if not MOCK and dev == "cuda": torch.cuda.synchronize()
    sec = time.time() - t
    np.save(path, e); log[str(k)] = {"sec": sec, "n": len(e), "device": gpu_name}
    json.dump(log, open(log_path, "w"))
    print(f"{k + len(e):>6,} / {len(texts):,}  ({sec:.1f}초)")
embed_sec = sum(v["sec"] for v in log.values())
E = np.concatenate([np.load(f"{OUT_DIR}/r3_emb_{k:06d}.npy") for k in range(0, len(texts), CHUNK)])
assert E.shape == (len(df), 768)
print("임베딩 완료:", E.shape, "| 계산 시간 합계", round(embed_sec, 1), "초 | 장치", gpu_name)

## 3단계: CPU로 하면 얼마나 걸릴까? (비용 비교용)

R0~R2는 GPU 없는 컴퓨터(4코어)에서 쟀다. 공정하게 비교하려고 CPU로 200개만 임베딩해 보고 1,000개당 시간을 추정한다.

In [ ]:
if MOCK:
    cpu_per_1k = None
else:
    cpu_model = AutoModel.from_pretrained(MODEL_NAME).eval()          # CPU, 32비트
    sample_idx = np.random.default_rng(0).choice(len(texts), 200, replace=False)
    t = time.time(); _ = embed([texts[i] for i in sample_idx], model=cpu_model, device="cpu", batch=16)
    cpu_per_1k = (time.time() - t) / 200 * 1000
    del cpu_model
    print(f"CPU({os.cpu_count()}코어)로 PR 1,000개 임베딩 추정: {cpu_per_1k:.1f}초")

## 4단계: R3 학습과 비교

임베딩 768개를 PCA로 128개로 줄인 뒤(R2와 같은 개수) R1 피처에 붙인다. 이게 R3다.

In [ ]:
from sklearn.decomposition import PCA
t = time.time(); pca = PCA(PCA_DIM, random_state=0).fit(E[tr_mask]); Z = pca.transform(E).astype(np.float32); pca_sec = time.time() - t
zc = [f"cb{i}" for i in range(PCA_DIM)]
tr = pd.concat([tr, pd.DataFrame(Z[tr_mask], columns=zc, index=tr.index)], axis=1)
te = pd.concat([te, pd.DataFrame(Z[~tr_mask], columns=zc, index=te.index)], axis=1)
print("PCA:", round(pca_sec, 1), "초 | 남긴 정보 비율:", round(float(pca.explained_variance_ratio_.sum()), 3))

res["R3"] = {}
for s in SEEDS:
    p, r = run(R1 + zc, s); res["R3"][str(s)] = r; preds[f"R3_p{s}"] = p
    print("R3 seed", s, "AUC", round(r["auc"], 4), "| 학습", round(r["train_sec"], 1), "초")

def boot(pa, pb, B=500, seed=0):
    rng = np.random.default_rng(seed); d = []
    for _ in range(B):
        i = rng.integers(0, len(y_te), len(y_te))
        d.append(roc_auc_score(y_te[i], pb[i]) - roc_auc_score(y_te[i], pa[i]))
    return [float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))]

def paired(a, b):
    per = [res[b][str(s)]["auc"] - res[a][str(s)]["auc"] for s in SEEDS]
    pa = preds[[f"{a}_p{s}" for s in SEEDS]].mean(axis=1).to_numpy(); pb = preds[[f"{b}_p{s}" for s in SEEDS]].mean(axis=1).to_numpy()
    return {"per_seed": per, "mean": float(np.mean(per)), "boot95_seedavg": boot(pa, pb)}

cmp = {"R3_minus_R1": paired("R1", "R3"), "R1_minus_R0": paired("R0", "R1")}
print("R3 - R1 AUC 차이:", [round(x, 4) for x in cmp["R3_minus_R1"]["per_seed"]], "| 95% 범위", [round(x, 4) for x in cmp["R3_minus_R1"]["boot95_seedavg"]])

## 5단계: 결과 저장

`r3_results.json`과 `r3_preds.parquet` 두 파일이 드라이브 폴더에 생긴다. **이 두 파일을 하랑이에게 보내면 끝.**

In [ ]:
def summ(runs, key):
    v = np.array([r[key] for r in runs.values()]); return {"mean": float(v.mean()), "min": float(v.min()), "max": float(v.max())}
n = len(df)
out = {
    "models": {k: {"auc": summ(v, "auc"), "prauc": summ(v, "prauc_reject"), "train_sec": summ(v, "train_sec"),
                   "pred_sec": summ(v, "pred_sec"), "runs": v} for k, v in res.items()},
    "compare": cmp,
    "embed": {"model": MODEL_NAME, "max_length": MAX_LENGTH, "pooling": "mean", "pca_dim": PCA_DIM,
              "pca_explained": float(pca.explained_variance_ratio_.sum()), "device": gpu_name,
              "embed_sec_total": embed_sec, "embed_sec_per_1k": embed_sec / n * 1000,
              "cpu_embed_sec_per_1k_est": cpu_per_1k, "cpu_count": os.cpu_count(), "pca_sec": pca_sec},
    "env": {"python": platform.python_version(), "xgboost": xgb.__version__, "sklearn": sklearn.__version__, "mock": MOCK},
    "n_train": int(tr_mask.sum()), "n_test": int((~tr_mask).sum()),
}
if not MOCK:
    import transformers
    out["env"].update({"torch": torch.__version__, "transformers": transformers.__version__})
json.dump(out, open(f"{OUT_DIR}/r3_results.json", "w"), ensure_ascii=False, indent=1)
preds.to_parquet(f"{OUT_DIR}/r3_preds.parquet", index=False)

print("| 표현 | AUC (5번 평균) | PR-AUC(거절) | 학습 시간 |")
print("| --- | --- | --- | --- |")
for k in ["R0", "R1", "R3"]:
    m = out["models"][k]
    print(f"| {k} | {m['auc']['mean']:.3f} ({m['auc']['min']:.3f}~{m['auc']['max']:.3f}) | {m['prauc']['mean']:.3f} | {m['train_sec']['mean']:.1f}초 |")
c = cmp["R3_minus_R1"]
print(f"\nR3 - R1: 평균 {c['mean']:+.4f}, 5번 범위 {min(c['per_seed']):+.4f}~{max(c['per_seed']):+.4f}, 95% 범위 {c['boot95_seedavg'][0]:+.4f}~{c['boot95_seedavg'][1]:+.4f}")
print(f"임베딩: {gpu_name}에서 PR 1,000개당 {out['embed']['embed_sec_per_1k']:.2f}초", "" if cpu_per_1k is None else f"| CPU 추정 {cpu_per_1k:.1f}초")
print("저장 완료:", f"{OUT_DIR}/r3_results.json", f"{OUT_DIR}/r3_preds.parquet")